In [1]:
from pathlib import Path
from rdflib import Graph

# 1. Thư mục dự án
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data" / "ACME_Insurance"

# 2. Các file đầu vào
DB_FILE = PROJECT_DIR / "results" / "sql" / "acme.duckdb"
ONTOLOGY_FILE = DATA_DIR / "ontology" / "insurance.ttl"
MAPPING_FILE = DATA_DIR / "data" / "data.world_P&C_Insurance_Ontology_V1.r2rml"
BENCHMARK_FILE = DATA_DIR / "investigation" / "acme-benchmark.ttl"

# 3. Thư mục lưu kết quả KG
KG_DIR = PROJECT_DIR / "results" / "kg"
KG_DIR.mkdir(parents=True, exist_ok=True)

# 4. Kiểm tra các file đầu vào
for file in [DB_FILE, ONTOLOGY_FILE, MAPPING_FILE, BENCHMARK_FILE]:
    print(file.name, ":", file.exists())

# 5. Đọc Ontology và R2RML Mapping
ontology = Graph()
ontology.parse(ONTOLOGY_FILE, format="turtle")

mapping = Graph()
mapping.parse(MAPPING_FILE, format="turtle")

print("\nOntology Triples:", len(ontology))
print("Mapping Triples:", len(mapping))

acme.duckdb : True
insurance.ttl : True
data.world_P&C_Insurance_Ontology_V1.r2rml : True
acme-benchmark.ttl : True

Ontology Triples: 164
Mapping Triples: 561


In [2]:
import pandas as pd
from rdflib import RDF, Namespace

RR = Namespace("http://www.w3.org/ns/r2rml#")

# 1. Lấy danh sách bảng CSV
csv_tables = {
    file.stem.lower()
    for file in (DATA_DIR / "data").glob("*.csv")
}

# 2. Kiểm tra từng Triples Map
results = []

for tm in mapping.subjects(RDF.type, RR.TriplesMap):
    logical = mapping.value(tm, RR.logicalTable)

    table = mapping.value(logical, RR.tableName)
    sql = mapping.value(logical, RR.sqlQuery)

    if table is not None:
        source = str(table).split(".")[-1]
        source_type = "tableName"
        status = "OK" if source.lower() in csv_tables else "MISSING"

    elif sql is not None:
        source = "SQL Query"
        source_type = "sqlQuery"
        status = "NEED CHECK"

    else:
        source = "Unknown"
        source_type = "Unknown"
        status = "NEED CHECK"

    results.append({
        "Mapping": str(tm).split("#")[-1],
        "Source": source,
        "Type": source_type,
        "Status": status
    })

# 3. Tổng hợp kết quả
mapping_df = pd.DataFrame(results)

print("Total mappings:", len(mapping_df))
print("\nMapping types:")
print(mapping_df["Type"].value_counts())

print("\nStatus:")
print(mapping_df["Status"].value_counts())

display(mapping_df)

Total mappings: 62

Mapping types:
Type
sqlQuery     39
tableName    23
Name: count, dtype: int64

Status:
Status
NEED CHECK    39
OK            23
Name: count, dtype: int64


,Mapping,Source,Type,Status
0,TripleMap_Agent_0,SQL Query,sqlQuery,NEED CHECK
1,TripleMap_hasclaim_8,claim_coverage,tableName,OK
2,TripleMap_hascatastrophe_19,claim,tableName,OK
3,TripleMap_PolicyCoverageEffectiveDate_19,policy_coverage_detail,tableName,OK
4,TripleMap_underwritespolicy_16,SQL Query,sqlQuery,NEED CHECK
...,...,...,...,...
57,TripleMap_ZipCode_10,SQL Query,sqlQuery,NEED CHECK
58,TripleMap_AddressLine2_16,SQL Query,sqlQuery,NEED CHECK
59,TripleMap_State_11,SQL Query,sqlQuery,NEED CHECK
60,TripleMap_PolicyEffectiveDate_15,policy,tableName,OK


In [3]:
import duckdb
import pandas as pd

# 1. Kết nối database đã tạo ở Experiment 02
conn = duckdb.connect(str(DB_FILE), read_only=True)

# 2. Lưu SQL gốc và kết quả kiểm tra
sql_sources = {}
sql_results = []

# 3. Duyệt 62 Triples Maps
for tm in mapping.subjects(RDF.type, RR.TriplesMap):
    logical = mapping.value(tm, RR.logicalTable)
    sql = mapping.value(logical, RR.sqlQuery)

    # Chỉ kiểm tra mappings có SQL Query
    if sql is None:
        continue

    mapping_name = str(tm).split("#")[-1]
    sql_text = str(sql)

    sql_sources[mapping_name] = sql_text

    # 4. Thử thực thi SQL
    try:
        if not sql_text.strip().lower().startswith(("select", "with")):
            raise ValueError("Not a SELECT query")

        rows = conn.execute(sql_text).fetchall()

        status = "OK"
        row_count = len(rows)
        error = ""

    except Exception as e:
        status = "ERROR"
        row_count = None
        error = str(e).splitlines()[0]

    # 5. Lưu kết quả
    sql_results.append({
        "Mapping": mapping_name,
        "Status": status,
        "Rows": row_count,
        "Error": error
    })

# 6. Tổng hợp
sql_check_df = pd.DataFrame(sql_results)

print("Total SQL Queries:", len(sql_check_df))

print("\nExecution Status:")
print(sql_check_df["Status"].value_counts())

print("\nFailed Queries:")
display(sql_check_df[sql_check_df["Status"] == "ERROR"])

Total SQL Queries: 39

Execution Status:
Status
OK    39
Name: count, dtype: int64

Failed Queries:


,Mapping,Status,Rows,Error


In [4]:
# 1. Nơi lưu dữ liệu đầu vào của từng mapping
source_data = {}
source_summary = []

# 2. Duyệt toàn bộ 62 Triples Maps
for tm in mapping.subjects(RDF.type, RR.TriplesMap):
    logical = mapping.value(tm, RR.logicalTable)

    table = mapping.value(logical, RR.tableName)
    sql = mapping.value(logical, RR.sqlQuery)

    # 3. Xác định nguồn dữ liệu
    if table is not None:
        table_name = str(table).split(".")[-1]
        query = f'SELECT * FROM "{table_name}"'
        source_type = "tableName"

    elif sql is not None:
        query = str(sql)
        source_type = "sqlQuery"

    else:
        raise ValueError(f"Missing logical table: {tm}")

    # 4. Đọc dữ liệu từ DuckDB
    df = conn.execute(query).df()

    # Lưu DataFrame theo Triples Map
    source_data[tm] = df

    # 5. Thống kê
    source_summary.append({
        "Mapping": str(tm).split("#")[-1],
        "Type": source_type,
        "Rows": len(df),
        "Columns": len(df.columns)
    })

# 6. Hiển thị kết quả
source_df = pd.DataFrame(source_summary)

print("Total mapping sources:", len(source_data))
print("\nSource types:")
print(source_df["Type"].value_counts())

print("\nEmpty sources:", (source_df["Rows"] == 0).sum())

display(source_df)

Total mapping sources: 62

Source types:
Type
sqlQuery     39
tableName    23
Name: count, dtype: int64

Empty sources: 0


,Mapping,Type,Rows,Columns
0,TripleMap_Agent_0,sqlQuery,1,1
1,TripleMap_hasclaim_8,tableName,2,3
2,TripleMap_hascatastrophe_19,tableName,2,14
3,TripleMap_PolicyCoverageEffectiveDate_19,tableName,6,9
4,TripleMap_underwritespolicy_16,sqlQuery,2,2
...,...,...,...,...
57,TripleMap_ZipCode_10,sqlQuery,8,9
58,TripleMap_AddressLine2_16,sqlQuery,8,9
59,TripleMap_State_11,sqlQuery,8,9
60,TripleMap_PolicyEffectiveDate_15,tableName,2,6


In [5]:
import re
import pandas as pd
from urllib.parse import quote
from rdflib import URIRef, Literal

# 1. Lấy giá trị cột, không phân biệt chữ hoa/thường
def get_value(row, column):
    for name, value in row.items():
        if name.lower() == column.lower():
            if pd.isna(value):
                return None
            return value

    raise KeyError(f"Column not found: {column}")


# 2. Tạo RDF Term từ một R2RML Term Map
def make_term(term_map, row):

    # Trường hợp 1: rr:template -> RDF IRI
    template = mapping.value(term_map, RR.template)

    if template is not None:
        url = str(template)

        for column in re.findall(r"\{([^{}]+)\}", url):
            value = get_value(row, column)

            if value is None:
                return None

            encoded = quote(str(value), safe="-._~")
            url = url.replace("{" + column + "}", encoded)

        return URIRef(url)

    # Trường hợp 2: rr:column -> RDF Literal
    column = mapping.value(term_map, RR.column)

    if column is not None:
        value = get_value(row, str(column))

        if value is None:
            return None

        # Chuyển kiểu dữ liệu Pandas về Python
        if isinstance(value, pd.Timestamp):
            value = value.to_pydatetime()
        elif hasattr(value, "item"):
            value = value.item()

        return Literal(value)

    raise ValueError(f"Unsupported term map: {term_map}")


# 3. Thử tạo RDF Subject và Object từ một mapping
example_tm = next(
    tm for tm in source_data
    if str(tm).endswith("TripleMap_hascatastrophe_19")
)

row = source_data[example_tm].to_dict("records")[0]

subject_map = mapping.value(example_tm, RR.subjectMap)
pom = mapping.value(example_tm, RR.predicateObjectMap)
object_map = mapping.value(pom, RR.objectMap)

subject = make_term(subject_map, row)
predicate = mapping.value(pom, RR.predicate)
object_ = make_term(object_map, row)

print("Subject:", subject)
print("Predicate:", predicate)
print("Object:", object_)
print("\nTerm types:", type(subject).__name__, type(object_).__name__)

Subject: https://myinsurancecompany.linked.data.world/d/omg-pc-database/Claim-1
Predicate: http://data.world/schema/insurance/hasCatastrophe
Object: https://myinsurancecompany.linked.data.world/d/omg-pc-database/Catastrophe-4

Term types: URIRef URIRef


In [6]:
from rdflib import Graph, RDF

# 1. Khởi tạo RDF Knowledge Graph
kg = Graph()
kg.bind("ins", "http://data.world/schema/insurance/")

kg_stats = []

# 2. Duyệt toàn bộ 62 R2RML Mappings
for tm, df in source_data.items():

    subject_map = mapping.value(tm, RR.subjectMap)

    # Các lớp RDF của Subject
    classes = list(mapping.objects(subject_map, RR["class"]))

    # Các quy tắc tạo Predicate và Object
    poms = list(mapping.objects(tm, RR.predicateObjectMap))

    before = len(kg)

    # 3. Duyệt từng dòng dữ liệu SQL
    for row in df.to_dict("records"):

        subject = make_term(subject_map, row)

        if subject is None:
            continue

        # 4. Tạo rdf:type từ rr:class
        for rdf_class in classes:
            kg.add((subject, RDF.type, rdf_class))

        # 5. Tạo các Predicate-Object Triples
        for pom in poms:

            object_map = mapping.value(pom, RR.objectMap)
            object_ = make_term(object_map, row)

            if object_ is None:
                continue

            for predicate in mapping.objects(pom, RR.predicate):
                kg.add((subject, predicate, object_))

    # 6. Lưu thống kê từng mapping
    kg_stats.append({
        "Mapping": str(tm).split("#")[-1],
        "Source Rows": len(df),
        "New Triples": len(kg) - before
    })

# 7. Xuất RDF Knowledge Graph
KG_FILE = KG_DIR / "acme_knowledge_graph.ttl"
kg.serialize(destination=str(KG_FILE), format="turtle")

# 8. Thống kê kết quả
kg_stats_df = pd.DataFrame(kg_stats)

print("Mappings processed:", len(kg_stats_df))
print("Total RDF Triples:", len(kg))
print("Unique Subjects:", len(set(kg.subjects())))
print("Unique Predicates:", len(set(kg.predicates())))
print("RDF Type Triples:", len(list(kg.triples((None, RDF.type, None)))))
print("Saved to:", KG_FILE)

print("\nMappings adding no new triples:")
display(kg_stats_df[kg_stats_df["New Triples"] == 0])

Mappings processed: 62
Total RDF Triples: 228
Unique Subjects: 51
Unique Predicates: 50
RDF Type Triples: 41
Saved to: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\kg\acme_knowledge_graph.ttl

Mappings adding no new triples:


,Mapping,Source Rows,New Triples
58,TripleMap_AddressLine2_16,8,0


In [11]:
from rdflib import Graph, RDF
import pandas as pd

# 1. Xuất Knowledge Graph sang định dạng N-Triples
KG_FILE = KG_DIR / "acme_knowledge_graph.nt"

kg.serialize(
    destination=str(KG_FILE),
    format="nt",
    encoding="utf-8"
)

# 2. Đọc lại Knowledge Graph và kiểm tra dữ liệu
kg_check = Graph()
kg_check.parse(str(KG_FILE), format="nt")

triples_preserved = set(kg) == set(kg_check)

print("=== KIỂM TRA RDF ===")
print("Original triples:", len(kg))
print("Saved triples:", len(kg_check))
print("Triples preserved:", triples_preserved)

assert triples_preserved, "RDF triples changed after serialization!"

# 3. Thống kê số lượng thực thể theo RDF Class
class_counts = {}

for subject, rdf_class in kg_check.subject_objects(RDF.type):
    class_name = str(rdf_class).split("/")[-1]
    class_counts[class_name] = class_counts.get(class_name, 0) + 1

class_df = pd.DataFrame([
    {"Class": name, "Count": count}
    for name, count in class_counts.items()
])

class_df = class_df.sort_values(
    "Count", ascending=False
).reset_index(drop=True)

print("\n=== THỐNG KÊ RDF CLASSES ===")
print("Total classes:", len(class_df))
display(class_df)

# 4. Kiểm tra mapping không tạo thêm RDF Triples
address_tm = next(
    tm for tm in source_data
    if str(tm).endswith("TripleMap_AddressLine2_16")
)

pom = mapping.value(address_tm, RR.predicateObjectMap)
object_map = mapping.value(pom, RR.objectMap)
column = str(mapping.value(object_map, RR.column))

address_rows = source_data[address_tm].to_dict("records")

non_null_count = sum(
    get_value(row, column) is not None
    for row in address_rows
)

print("\n=== KIỂM TRA ADDRESS LINE 2 ===")
print("Source rows:", len(address_rows))
print("Non-null values:", non_null_count)

# 5. Thực thi thử một truy vấn SPARQL trên Knowledge Graph
test_query = """
PREFIX ins: <http://data.world/schema/insurance/>

SELECT ?claim ?catastrophe
WHERE {
    ?claim ins:hasCatastrophe ?catastrophe .
}
ORDER BY ?claim
LIMIT 5
"""

sparql_results = kg_check.query(test_query)

test_df = pd.DataFrame([
    {
        "Claim": str(row.claim).split("/")[-1],
        "Catastrophe": str(row.catastrophe).split("/")[-1]
    }
    for row in sparql_results
], columns=["Claim", "Catastrophe"])

print("\n=== KẾT QUẢ TRUY VẤN SPARQL ===")
print("Returned rows:", len(test_df))
display(test_df)

# 6. Tổng hợp kết quả Knowledge Graph
print("\n=== TỔNG KẾT KNOWLEDGE GRAPH ===")
print("Total triples:", len(kg_check))
print("Unique subjects:", len(set(kg_check.subjects())))
print("Unique predicates:", len(set(kg_check.predicates())))
print("RDF classes:", len(class_df))
print("Validation:", "PASSED" if triples_preserved else "FAILED")
print("Saved file:", KG_FILE)

=== KIỂM TRA RDF ===
Original triples: 228
Saved triples: 228
Triples preserved: True

=== THỐNG KÊ RDF CLASSES ===
Total classes: 12


,Class,Count
0,Location,8
1,Premium,6
2,PolicyCoverageDetail,6
3,PolicyAmount,6
4,Catastrophe,4
5,Claim,2
6,UnderwritingAssessment,2
7,InsurableObject,2
8,Policy,2
9,Agent,1



=== KIỂM TRA ADDRESS LINE 2 ===
Source rows: 8
Non-null values: 0

=== KẾT QUẢ TRUY VẤN SPARQL ===
Returned rows: 2


,Claim,Catastrophe
0,Claim-1,Catastrophe-4
1,Claim-2,Catastrophe-4



=== TỔNG KẾT KNOWLEDGE GRAPH ===
Total triples: 228
Unique subjects: 51
Unique predicates: 50
RDF classes: 12
Validation: PASSED
Saved file: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\kg\acme_knowledge_graph.nt


In [12]:
import re
import pandas as pd
from rdflib import Graph, Namespace, RDF
from rdflib.plugins.sparql.parser import parseQuery

# 1. Đọc benchmark chứa các câu hỏi và SPARQL Reference
benchmark = Graph()
benchmark.parse(BENCHMARK_FILE, format="turtle")

QA = Namespace("http://models.data.world/benchmarks/QandA#")
DWT = Namespace("https://templates.data.world/")

# 2. Lưu truy vấn SPARQL gốc và phiên bản chạy local
sparql_references = {}
sparql_results = []

# 3. Duyệt tất cả câu hỏi trong benchmark
inquiries = sorted(
    benchmark.subjects(RDF.type, QA.Inquiry),
    key=str
)

for inquiry in inquiries:
    question = str(benchmark.value(inquiry, QA.prompt))

    # Lấy các SPARQL Reference của câu hỏi
    for ref in benchmark.objects(inquiry, QA.expects):

        if (ref, RDF.type, DWT.SparqlQuery) not in benchmark:
            continue

        ref_id = str(ref)
        original_sql = str(benchmark.value(ref, QA.queryText))

        # 4. Bỏ SERVICE :mapped nhưng giữ nguyên các dấu ngoặc
        local_sql, replacements = re.subn(
            r"\bSERVICE\s+[\w-]*:mapped\s*(?=\{)",
            "",
            original_sql,
            flags=re.IGNORECASE
        )

        # 5. Kiểm tra cú pháp SPARQL sau khi chuẩn hóa
        try:
            parseQuery(local_sql)
            status = "OK"
            error = ""
        except Exception as e:
            status = "ERROR"
            error = str(e).splitlines()[0]

        # 6. Lưu cả truy vấn gốc và phiên bản local
        sparql_references[ref_id] = {
            "question": question,
            "original": original_sql,
            "local": local_sql
        }

        sparql_results.append({
            "Question": question,
            "Reference ID": ref_id,
            "SERVICE Removed": replacements,
            "Status": status,
            "Error": error
        })

# 7. Tổng hợp kết quả
sparql_check_df = pd.DataFrame(sparql_results)

print("Total SPARQL References:", len(sparql_check_df))
print("Total SERVICE removed:", sparql_check_df["SERVICE Removed"].sum())

print("\nSyntax Status:")
print(sparql_check_df["Status"].value_counts())

print("\nFailed Queries:")
display(sparql_check_df[sparql_check_df["Status"] == "ERROR"])

Total SPARQL References: 44
Total SERVICE removed: 44

Syntax Status:
Status
OK    44
Name: count, dtype: int64

Failed Queries:


,Question,Reference ID,SERVICE Removed,Status,Error


In [16]:
import time
import pandas as pd
from rdflib import Graph, URIRef, Namespace

INS = Namespace("http://data.world/schema/insurance/")

# 1. Tạo bản sao Knowledge Graph để chuẩn hóa
kg_local = Graph()
kg_local += kg_check

# 2. Sửa URI Policy Holder không nhất quán trong R2RML gốc
holder_subjects = set(
    kg_local.subjects(INS.policyHolderId, None)
)

fixed_links = 0

for policy, holder in list(kg_local.subject_objects(INS.hasPolicyHolder)):
    if "/PolicyHolder-" not in str(holder):
        continue

    correct_holder = URIRef(
        str(holder).replace("/PolicyHolder-", "/Policy-Holder-")
    )

    if correct_holder not in holder_subjects:
        raise ValueError(f"Policy Holder not found: {correct_holder}")

    kg_local.remove((policy, INS.hasPolicyHolder, holder))
    kg_local.add((policy, INS.hasPolicyHolder, correct_holder))

    fixed_links += 1

print("Policy Holder links fixed:", fixed_links)

# 3. Chuẩn bị nơi lưu kết quả SPARQL
sparql_execution_log = []
sparql_outputs = {}
local_sparql_queries = {}

date_fixes = 0

# 4. Duyệt toàn bộ 44 SPARQL Reference
for ref_id, data in sparql_references.items():

    question = data["question"]
    query = data["local"]

    # Sửa điều kiện lọc ngày trong truy vấn Claim năm 2019
    if "query-97c94a18" in ref_id:
        old_filter = (
            "FILTER(?ClaimCloseDate >= '2019-01-01' "
            "&& ?ClaimCloseDate <= '2019-12-31')"
        )
        new_filter = "FILTER(YEAR(?ClaimCloseDate) = 2019)"

        assert old_filter in query, "Unexpected date filter"

        query = query.replace(old_filter, new_filter)
        date_fixes += 1

    # Lưu truy vấn thực tế được thực thi
    local_sparql_queries[ref_id] = query

    # 5. Thực thi SPARQL trên Knowledge Graph local
    start_time = time.perf_counter()

    try:
        result = kg_local.query(query)

        if result.type != "SELECT":
            raise ValueError(f"Unsupported query type: {result.type}")

        columns = [str(v) for v in result.vars]

        df = pd.DataFrame([
            {
                col: str(row[i]) if row[i] is not None else None
                for i, col in enumerate(columns)
            }
            for row in result
        ], columns=columns)

        sparql_outputs[ref_id] = df

        status = "OK"
        row_count = len(df)
        error = ""

    except Exception as e:
        status = "ERROR"
        row_count = None
        error = str(e).replace("\n", " ")[:300]

    elapsed = time.perf_counter() - start_time

    # 6. Ghi nhận kết quả thực thi
    sparql_execution_log.append({
        "Question": question,
        "Reference ID": ref_id,
        "Status": status,
        "Rows": row_count,
        "Time (s)": round(elapsed, 4),
        "Error": error
    })

# 7. Tổng hợp kết quả
sparql_log_df = pd.DataFrame(sparql_execution_log)

print("\nTotal SPARQL References:", len(sparql_log_df))
print("Date filters fixed:", date_fixes)
print("Total RDF Triples:", len(kg_local))

print("\nExecution Status:")
print(sparql_log_df["Status"].value_counts())

print("\nEmpty Results:")
print((sparql_log_df["Rows"] == 0).sum())

print("\nFailed Queries:")
display(sparql_log_df[sparql_log_df["Status"] == "ERROR"])

print("\nEmpty Queries:")
display(sparql_log_df[sparql_log_df["Rows"] == 0])

Policy Holder links fixed: 2

Total SPARQL References: 44
Date filters fixed: 1
Total RDF Triples: 228

Execution Status:
Status
OK    44
Name: count, dtype: int64

Empty Results:
0

Failed Queries:


,Question,Reference ID,Status,Rows,Time (s),Error



Empty Queries:


,Question,Reference ID,Status,Rows,Time (s),Error


In [18]:
import pandas as pd

# 1. Tạo thư mục lưu kết quả SPARQL
SPARQL_DIR = KG_DIR / "sparql_results"
SPARQL_DIR.mkdir(parents=True, exist_ok=True)

# 2. Xuất Knowledge Graph đã chuẩn hóa
KG_LOCAL_FILE = KG_DIR / "acme_knowledge_graph_local.nt"

kg_local.serialize(
    destination=str(KG_LOCAL_FILE),
    format="nt",
    encoding="utf-8"
)

# 3. Lưu kết quả của từng SPARQL Reference
for ref_id, df in sparql_outputs.items():
    filename = ref_id.split("/")[-1] + ".csv"
    df.to_csv(SPARQL_DIR / filename, index=False)

# 4. Lưu các truy vấn SPARQL gốc và phiên bản local
reference_rows = []

for ref_id, data in sparql_references.items():
    reference_rows.append({
        "Question": data["question"],
        "Reference ID": ref_id,
        "Original SPARQL": data["original"],
        "Local SPARQL": local_sparql_queries[ref_id]
    })

reference_df = pd.DataFrame(reference_rows)

reference_df.to_csv(
    KG_DIR / "sparql_references.csv",
    index=False,
    encoding="utf-8-sig"
)

# 5. Lưu log thực thi SPARQL
sparql_log_df.to_csv(
    KG_DIR / "sparql_execution_log.csv",
    index=False
)

# 6. Đọc kết quả SQL Reference từ Experiment 02
sql_log_file = PROJECT_DIR / "results" / "sql" / "sql_execution_log.csv"
sql_df = pd.read_csv(sql_log_file)

# Tổng hợp SQL Reference theo từng câu hỏi
sql_summary = sql_df.groupby("Question", as_index=False).agg(
    SQL_References=("Reference ID", "count"),
    SQL_Successful=("Status", lambda x: (x == "OK").sum()),
    SQL_Rows=("Rows", lambda x: ", ".join(
        str(int(v)) for v in x.dropna()
    ))
)

# 7. Đối chiếu kết quả thực thi SQL và SPARQL
comparison_df = sparql_log_df[
    ["Question", "Reference ID", "Status", "Rows"]
].rename(columns={
    "Reference ID": "SPARQL Reference ID",
    "Status": "SPARQL Status",
    "Rows": "SPARQL Rows"
})

comparison_df = comparison_df.merge(
    sql_summary,
    on="Question",
    how="left"
)

comparison_df.to_csv(
    KG_DIR / "sql_sparql_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

# 8. Tổng hợp kết quả cuối cùng
print("=== TỔNG KẾT EXPERIMENT 03 ===")
print("R2RML Mappings:", len(source_data))
print("Original RDF Triples:", len(kg_check))
print("Local RDF Triples:", len(kg_local))

print("\nSPARQL References:", len(sparql_log_df))
print("Successful:", (sparql_log_df["Status"] == "OK").sum())
print("Failed:", (sparql_log_df["Status"] == "ERROR").sum())
print("Empty Results:", (sparql_log_df["Rows"] == 0).sum())

print("\nSQL References:", len(sql_df))
print("Questions matched with SQL:", comparison_df["SQL_References"].notna().sum())

print("\nSaved SPARQL Results:", len(sparql_outputs))
print("Output directory:", KG_DIR)

# 9. Hiển thị bảng đối chiếu
display(comparison_df)

=== TỔNG KẾT EXPERIMENT 03 ===
R2RML Mappings: 62
Original RDF Triples: 228
Local RDF Triples: 228

SPARQL References: 44
Successful: 44
Failed: 0
Empty Results: 0

SQL References: 45
Questions matched with SQL: 44

Saved SPARQL Results: 44
Output directory: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\kg


,Question,SPARQL Reference ID,SPARQL Status,SPARQL Rows,SQL_References,SQL_Successful,SQL_Rows
0,What is the loss ratio of each policy and agen...,https://templates.data.world/query-a916ea29-b4...,OK,1,1,1,1
1,Return all the policies we have by policy numb...,https://templates.data.world/query-481d808c-a3...,OK,2,1,1,2
2,"What are the loss payment, loss reserve, expen...",https://templates.data.world/query-a7e183f5-33...,OK,2,1,1,2
3,"What are the total loss, which is the sum of l...",https://templates.data.world/query-d1fc6863-b4...,OK,1,1,1,1
4,Return agents and the policy they have sold th...,https://templates.data.world/query-61bca3dc-7a...,OK,2,1,1,2
5,What is the average policy size which is the t...,https://templates.data.world/query-1229101a-9a...,OK,1,1,1,1
6,"What are the total loss, which is the sum of l...",https://templates.data.world/query-0d8c074f-3c...,OK,2,1,1,2
7,What is the total amount of premiums that a po...,https://templates.data.world/query-ae1e0b7b-7f...,OK,2,1,1,2
8,"What are the loss payment, loss reserve, expen...",https://templates.data.world/query-aebf3687-c8...,OK,2,1,1,2
9,Return agents and the policy they have sold th...,https://templates.data.world/query-3d645ba9-0c...,OK,2,1,1,2
